# Deferred elements

Deferred elements have a type that start with @

This will make replacement dicts, setting these to be drifts.

These must be bootstrapped with already converted Bmad models. 

In [1]:
from pytao import Tao
import json
import os

## Extra deferred

These are missing electronics. Will further mark as deferred.

In [2]:
EXTRA_DEFERED = []


## Devel

In [3]:
tao = Tao('-init $LCLS_LATTICE/bmad/models/cu_sxr/tao.init -noplot')

In [4]:
IXLIST = tao.lat_list('*', 'ele.ix_ele', flags='-array_out -index_order -no_slaves')

heads = [tao.ele_head(i) for i in IXLIST]

deferred = [head['type'].startswith('@') for head in heads]

ix_deferred = IXLIST[deferred]


In [5]:
tao.ele_head(346)['type'].startswith('@')

False

In [6]:
def is_deferred(head):
    if head['type'].startswith('@'):
        deferred = True
    elif head['name'] in EXTRA_DEFERED:
        deferred = True
    else:
        deferred = False
    return deferred

In [7]:
def ele_to_drift(i):
    d = tao.ele_gen_attribs(i)
    d.update(tao.ele_head(i))
    
    name = d['name']
    if 'L' in d:
        L = d['L']
    else:
        L = 0
    
    type = d['type']
    if not type.startswith('@'):
        print('Warning: no @')
        type = '@'+type
    
    key = d['key'].upper()
    
    # Handle the half quad, sex length convention. 
    if key in ['QUADRUPOLE', 'SEXTUPOLE']:
        L = L/2
    
    line = f'{name}: drift, L = {L}, type ="{type}", descrip = "deferred {key}" '

    return {name:line}

def deferred_replacements():
    ixlist = tao.lat_list('*', 'ele.ix_ele', flags='-array_out -index_order -no_slaves')
    heads = [tao.ele_head(i) for i in ixlist]
    deferred = list(map(is_deferred, heads))
    ix_deferred = ixlist[deferred]
    
    replacements = {}
    for ix in ix_deferred:
        replacements.update(ele_to_drift(ix))
    return replacements

ele_to_drift(1)

{'DL00': 'DL00: drift, L = -0.869048, type ="@", descrip = "deferred DRIFT" '}

In [8]:
deferred_replacements()

{'BXKIK': 'BXKIK: drift, L = 1.0601, type ="@9,1.92K41.2", descrip = "deferred SBEND" ',
 'RFBBP33': 'RFBBP33: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'SDL1': 'SDL1: drift, L = 0.05, type ="@1,1.38S3.00", descrip = "deferred SEXTUPOLE" ',
 'SDL2': 'SDL2: drift, L = 0.05, type ="@1,1.38S3.00", descrip = "deferred SEXTUPOLE" ',
 'PCTDKIK3S': 'PCTDKIK3S: drift, L = 0.8128, type ="@5,BCS PC", descrip = "deferred ECOLLIMATOR" ',
 'PCTDKIK4S': 'PCTDKIK4S: drift, L = 0.8128, type ="@5,BCS PC", descrip = "deferred ECOLLIMATOR" ',
 'RFBEM4B': 'RFBEM4B: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFBE32B': 'RFBE32B: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFBE34B': 'RFBE34B: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFBE36B': 'RFBE36B: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'TRUE1B': 'TRUE1B: drift, L = 0.0, type ="@5", descrip = "deferred INSTR

# Process all models

In [9]:
cu_models = [f for f in os.listdir('../models') if f.startswith('cu_')]
cu_models.remove('cu_inj')
sc_models = [f for f in os.listdir('../models') if f.startswith('sc_')]

cu_models, sc_models

(['cu_sxr', 'cu_spec', 'cu_hxr', 'cu_linac'],
 ['sc_diag0', 'sc_dasel', 'sc_sxr', 'sc_hxr', 'sc_bsyd'])

In [10]:
def get_all_replacements(models, file=None):
    replacements = {}
    for name in models:
        print(name)
        tao = Tao(f'-init $LCLS_LATTICE/bmad/models/{name}/tao.init -noplot')
        replacements.update(deferred_replacements())
        
    if file:
        with open(file, 'w') as outfile:
            json.dump(replacements, outfile, ensure_ascii=True, indent='  ')
        print('written:', file)
    return replacements

get_all_replacements(cu_models, 'replacements/deferred_cu_replacements.json')


cu_sxr
cu_spec
cu_hxr
cu_linac
written: replacements/deferred_cu_replacements.json


{'BXKIK': 'BXKIK: drift, L = 1.0601, type ="@9,1.92K41.2", descrip = "deferred SBEND" ',
 'RFBBP33': 'RFBBP33: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'SDL1': 'SDL1: drift, L = 0.05, type ="@1,1.38S3.00", descrip = "deferred SEXTUPOLE" ',
 'SDL2': 'SDL2: drift, L = 0.05, type ="@1,1.38S3.00", descrip = "deferred SEXTUPOLE" ',
 'PCTDKIK3S': 'PCTDKIK3S: drift, L = 0.8128, type ="@5,BCS PC", descrip = "deferred ECOLLIMATOR" ',
 'PCTDKIK4S': 'PCTDKIK4S: drift, L = 0.8128, type ="@5,BCS PC", descrip = "deferred ECOLLIMATOR" ',
 'RFBEM4B': 'RFBEM4B: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFBE32B': 'RFBE32B: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFBE34B': 'RFBE34B: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFBE36B': 'RFBE36B: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'TRUE1B': 'TRUE1B: drift, L = 0.0, type ="@5", descrip = "deferred INSTR

In [11]:
# SC replacements
get_all_replacements(sc_models, 'replacements/deferred_sc_replacements.json')

sc_diag0
sc_dasel
sc_sxr
sc_hxr
sc_bsyd
written: replacements/deferred_sc_replacements.json


{'RFB0H00': 'RFB0H00: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFB0H04': 'RFB0H04: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFB0H08': 'RFB0H08: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFBHD00': 'RFBHD00: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFBHD04': 'RFBHD04: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'RFBDG001': 'RFBDG001: drift, L = 0.0, type ="@2,CavityL-1", descrip = "deferred MONITOR" ',
 'YCDGTCY': 'YCDGTCY: drift, L = 0.0, type ="@4,class-1t", descrip = "deferred VKICKER" ',
 'XCDGTCY': 'XCDGTCY: drift, L = 0.0, type ="@4,class-1t", descrip = "deferred HKICKER" ',
 'OTRDG01': 'OTRDG01: drift, L = 0.0, type ="@3,YAG/OTR PSI", descrip = "deferred MONITOR" ',
 'RFBDG002': 'RFBDG002: drift, L = 0.0, type ="@2,CavityS-1", descrip = "deferred MONITOR" ',
 'OTRDG03': 'OTRDG03: drift, L = 0.0, type ="@3,YAG/OTR PSI", descrip = "def